<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 3.5: Object Oriented Programming
**Prev: [Functional Programming](3.4_functional_programming.ipynb)**<br>
**Next: [Types](3.6_types.ipynb)**

## Motivation
Scala and Chisel are object-oriented programming languages, meaning code may be compartmentalized into objects.
Scala, which is built on Java, inherits many of Java's object-oriented features.
However, as we'll see below, there are some differences.
Chisel's hardware modules are similar to Verilog's modules, in that they can be instantiated and wired up as single or multiple instances.

## Setup

In [6]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

path: String = "/home/hheile/Uni/ESD-Project/Code/chisel-bootcamp/source/load-ivy.sc"

In [7]:
import chisel3._
import chisel3.util._
import chisel3.experimental._
import chisel3.tester._
import chisel3.tester.RawTester.test

import chisel3._

import chisel3.util._

import chisel3.experimental._

import chisel3.tester._

import chisel3.tester.RawTester.test

---
# Object Oriented Programming
This section outlines how Scala implements the object-oriented programming paradigm. So far you have already seen classes, but Scala also has the following features:
- Abstract classes
- Traits
- Objects
- Companion Objects
- Case Classes

## Abstract Classes
Abstract classes are just like other programming language implementations. They can define many unimplemented values that subclasses must implement. Any object can only directly inherit from one parent abstract class.

<span style="color:blue">**Example: Abstract Class**</span><br>

In [8]:
// abstract class: declares many unimplemented values that subclasses then must implement
// generally not needed much, as traits are very powerful; useful if constructor arguments are needed or the class is called by java code
abstract class MyAbstractClass {
  def myFunction(i: Int): Int       // general function definition
  val myValue: String               // class member
}

// subclass of the abstract class, a class can only inherit from one parent class
class ConcreteClass extends MyAbstractClass {
  def myFunction(i: Int): Int = i + 1           // implements function properly
  val myValue = "Hello World!"                  // defines class member
}

// Legal instantiation!
val concreteClass = new ConcreteClass()      
println(concreteClass.myValue)
println(concreteClass.myFunction(4))


Hello World!
5


defined class MyAbstractClass
defined class ConcreteClass
concreteClass: ConcreteClass = ammonite.$sess.cmd7$Helper$ConcreteClass@618dbd29

In [8]:
val abstractClass = new MyAbstractClass()   // Illegal! Cannot instantiate an abstract class

cmd8.sc:1: class MyAbstractClass is abstract; cannot be instantiated
val abstractClass = new MyAbstractClass()   // Illegal! Cannot instantiate an abstract class
                    ^Compilation Failed

: 

## Traits
Traits are very similar to abstract classes in that they can define unimplemented values. However, they differ in two ways:
- a class can inherit from multiple traits
- a trait cannot have constructor parameters (in Scala 2, in Scala 3 that has been added)

<span style="color:blue">**Example: Traits and Multiple Inheritance**</span><br>
Traits are how Scala implements multiple inheritance, as shown in the example below. `MyClass` extends from both traits `HasFunction` and `HasValue`:

In [9]:
// Similar to an abstract class but cannot have constructor parameters (at least not in Scala 2 which is used here, Scala 3 offers this)
// The rule of thumb (for Scala 3) is to use classes whenever you want to create instances of a particular type and enforce single-inheritance restrictions, 
// and traits when you want to decompose and reuse behaviour. Generally Traits should be preferred.
trait HasFunction {
  def myFunction(i: Int): Int   // general function definition
}

trait HasValue {
  val myValue: String           // general trait member
  val myOtherValue = 100        // concrete trait value
}

// Traits offer multiple inheritance: A class can implement many traits via "... extends Trait1 with Trait2 with Trait3..."
class MyClass extends HasFunction with HasValue {
  override def myFunction(i: Int): Int = i + 1
  val myValue = "Hello World!"
}

// Legal instantiation!
val myClass = new MyClass()
println(myClass.myValue)
println(myClass.myFunction(myClass.myOtherValue))

Hello World!
101


defined trait HasFunction
defined trait HasValue
defined class MyClass
myClass: MyClass = ammonite.$sess.cmd8$Helper$MyClass@1fe94e2f

In [9]:
val myTraitFunction = new HasFunction()     // Illegal! Cannot instantiate a trait

cmd9.sc:1: trait HasFunction is abstract; cannot be instantiated
val myTraitFunction = new HasFunction()     // Illegal! Cannot instantiate a trait
                      ^Compilation Failed

: 

In [9]:
val myTraitValue = new HasValue()           // Illegal! Cannot instantiate a trait

cmd9.sc:1: trait HasValue is abstract; cannot be instantiated
val myTraitValue = new HasValue()           // Illegal! Cannot instantiate a trait
                   ^Compilation Failed

: 

To inherit multiple traits, chain them like 

```scala
class MyClass extends HasTrait1 with HasTrait2 with HasTrait3 ...
```
In general, **always use traits over abstract classes**, unless you are certain you want to enforce the single-inheritance restriction of abstract classes.

## Objects
Scala has a language feature for these singleton classes, called objects. You cannot instantiate an object **(no need to call `new`)**; you can simply directly reference it. That makes them similar to Java static classes.

<span style="color:blue">**Example: Objects**</span><br>

In [10]:
// Objects are singleton classes (only a single instance exists)
// similar zu java static classes
object MyObject {
  val hi = "Hello World"
  def ask: String = "How are you?"   // function that returns the String
  def apply(msg: String) = msg  
  def calculate(i: Int) = i + 1
}

// singleton => no need to call "new", just directly reference it
println(MyObject.hi)
println(MyObject.ask)

// The apply method makes object callable like a function, which is why here you can give the parameter directly to the object
println(MyObject("This message is important!"))     // equivalent to MyObject.apply(msg)
println(MyObject.calculate(4))                      // equivalent to MyObject.apply(4)

Hello World
How are you?
This message is important!
5


defined object MyObject

## Companion Objects

When a class and an object share the same name and defined in the same file, the object is called a **companion object**. When you use `new` before the class/object name, it will instantiate the class. If you don't use `new`, it will reference the object:

<span style="color:blue">**Example: Companion Object**</span><br>

In [ ]:
// class with same name and in same file as object (see below) => companion class to object
class Lion {
    def roar(): Unit = println("I'M A CLASS!")
}

// object with same name and in same file as class (see above) => companion object to class
// put class related constants here or use to augment construction of the class 
// Especially interesting for factory methods!!!
object Lion {
    def roar(): Unit = println("I'M AN OBJECT!")
}

new Lion().roar()   // "new" instatiates the class
Lion.roar()         // direct access references the singleton object

I'M A CLASS!
I'M AN OBJECT!


defined object Lion
defined class Lion

Companion objects are usually used for the following reasons:
  1. to contain constants related to the class
  2. to execute code before/after the class constructor
  3. to create multiple constructors for a class

In the example below, we will instantiate a number of instances of Animal. We want each animal to have a name, and to know its order within all instantiations. Finally, if no name is given, it should get a default name.

In [13]:
object Animal {
    val defaultName = "Bigfoot"                 // constant of the default name for the class animal
    private var numberOfAnimals = 0             // private mutable counter to keep track of instance order
    
    def apply(name: String): Animal = {         // call Object with a name
        numberOfAnimals += 1
        new Animal(name, numberOfAnimals)       // instantiates class Animal => this apply is a factory method! Last line of this block => return value
    }
    
    def apply(): Animal = apply(defaultName)    // call Object just with parentheses => call the other apply with defaultName
}

class Animal(name: String, order: Int) {
  def info: String = s"Hi my name is $name, and I'm $order in line!"
}

val bunny = Animal.apply("Hopper")      // Calls the Animal factory method in the object
println(bunny.info)                     // Calls the function in the class
val cat = Animal("Whiskers")
println(cat.info)
val yeti = Animal()                     // Default name case
println(yeti.info)


Hi my name is Hopper, and I'm 1 in line!
Hi my name is Whiskers, and I'm 2 in line!
Hi my name is Bigfoot, and I'm 3 in line!


defined object Animal
defined class Animal
bunny: Animal = ammonite.$sess.cmd12$Helper$Animal@6f692d60
cat: Animal = ammonite.$sess.cmd12$Helper$Animal@40cccbcf
yeti: Animal = ammonite.$sess.cmd12$Helper$Animal@2916e849

*What's happening here?*
Our **Animal companion object** defines a constant relevant to ```class Animal```:
```scala
val defaultName = "Bigfoot"
```
It also defines a private mutable integer to keep track of the order of Animal instances:
```scala 
private var numberOfAnimals = 0
```
It defines two **apply** methods, which are known as **factory methods** in that they return instances of the **class Animal**. 
- The first creates an instance of Animal using only one argument, ```name```, and uses ```numberOfAnimals``` as well to call the Animal class constructor.
- The second factory method requires no argument, and instead uses the default name to call the other apply method.
```scala
def apply(name: String): Animal = {
            numberOfAnimals += 1
            new Animal(name, numberOfAnimals)
}
def apply(): Animal = apply(defaultName)
```
These factory methods can be called naively like this
```scala
val bunny = Animal.apply("Hopper")
```
which eliminates the need to use the new keyword, but the real magic is that the compiler assumes the apply method any time it sees parentheses applied to an instance or object:
```scala
val cat = Animal("Whiskers")
val yeti = Animal()
```
Factory methods, usually provided via companion objects, allow alternative ways to express instance creations, provide additional tests for constructor parameters, conversions, and eliminate the need to use the keyword ```new```. Note that you must call the companion object's `apply` method for `numberOfAnimals` to be incremented.

**Chisel uses many companion objects, like Module.** When you write the following:
```scala
val myModule = Module(new MyModule)
```
you are calling the **Module companion object**, so Chisel can run background code before and after instantiating 
```MyModule```.

## Case Classes

Case classes are a special type of Scala class that provides some cool additional features. They are very common in Scala programming, so this section outlines some of their useful features:
- Allows **external access** to the **class parameters**
- **Eliminates** the need to use **`new`** when instantiating the class
- Automatically creates an **unapply method** that supplies access to all of the class Parameters. (See section 3.6)
- Cannot be subclassed from

In the following example, we declare three different classes, `Nail`, `Screw`, and `Staple`.

In [14]:
class Nail(length: Int)                 // Regular class
val nail = new Nail(10)                 // Requires the `new` keyword

defined class Nail
nail: Nail = ammonite.$sess.cmd13$Helper$Nail@232beac8

In [14]:

println(nail.length)                 // Illegal! Class constructor parameters are not by default externally visible

cmd14.sc:1: value length is not a member of cmd14.this.cmd13.Nail
val res14 = println(nail.length)                 // Illegal! Class constructor parameters are not by default externally visible
                         ^Compilation Failed

: 

In [15]:
class Screw(val threadSpace: Int)       // By using the `val` keyword, threadSpace is now externally visible
val screw = new Screw(2)                // Requires the `new` keyword
println(screw.threadSpace)

2


defined class Screw
screw: Screw = ammonite.$sess.cmd14$Helper$Screw@713a1ad

In [ ]:
// Case classes cannot be extended by other classes aka other classes cannot inherit from them
case class Staple(isClosed: Boolean)    // Case class constructor parameters are, by default, externally visible (no val needed)
val staple = Staple(false)              // No `new` keyword required, because Scala automatically creates a companion object for case classes
println(staple.isClosed)

false


defined class Staple
staple: Staple = Staple(false)

`Nail` is a regular class, and its parameters are not externally visible because we did not use the `val` keyword in the argument list. It also requires the `new` keyword when declaring an instance of `Nail`.

`Screw` is declared similarly to `Nail`, but includes `val` in the argument list. This allows its parameter, `threadSpace`, to be visible externally.

By using a case class, `Staple` gets the benefit of all its parameters being externally visible (without needing the `val` keyword). In addition, `Staple` does not require using `new` when declaring a case class. This is because the Scala compiler automatically creates a companion object for every case class in your code, which contains an apply method for the case class.

Case classes are nice containers for generators with lots of parameters.
The constructor gives you a good place to define derived parameters and validate input.

In [17]:
case class SomeGeneratorParameters(
    someWidth: Int,
    someOtherWidth: Int = 10,
    pipelineMe: Boolean = false
) {
    require(someWidth >= 0)
    require(someOtherWidth >= 0)
    val totalWidth = someWidth + someOtherWidth
}

defined class SomeGeneratorParameters

---
# Inheritance with Chisel<a name="super"></a>
You've seen `Module`s and `Bundle`s before, but it's important to realize what's really going on.
Every Chisel module you make is a class extending the base type `Module`.
Every Chisel IO you make is a class extending the base type `Bundle` (or, in some special cases, `Bundle`'s supertype [`Record`](https://github.com/freechipsproject/chisel3/blob/v3.0.0/chiselFrontend/src/main/scala/chisel3/core/Aggregate.scala#L415)).
Chisel hardware types like `UInt` or `Bundle` all have `Data` as a supertype.
We'll explore using object oriented programming to create hierarchical hardware blocks and explore object reuse. You'll learn more about types and `Data` in the next Module on type generic generators.

## Module<a name="module"></a>
Whenever you want to create a hardware object in Chisel, it needs to have `Module` as a superclass.
Inheritance might not always be the right tool for reuse ([composition over inheritance](https://en.wikipedia.org/wiki/Composition_over_inheritance) is a common principle), but inheritance is still a powerful tool.
Below is an example of creating a `Module` and connecting multiple instantiations of them together hierarchically.

<span style="color:blue">**Example: Gray Encoder and Decoder**</span><br>
We'll create a hardware Gray encoder/decoder. The encode or decode operation choice is hardware programmable.

> "The reflected binary code (RBC), also known as reflected binary (RB) or Gray code after Frank Gray, is an ordering of the binary numeral system such that two successive values differ in only one bit (binary digit).
> 
> For example, the representation of the decimal value "1" in binary would normally be "001", and "2" would be "010". In Gray code, these values are represented as "001" and "011". That way, incrementing a value from 1 to 2 requires only one bit to change, instead of two.
> 
> Gray codes are widely used to prevent spurious output from electromechanical switches and to facilitate error correction in digital communications such as digital terrestrial television and some cable TV systems. The use of Gray code in these devices helps simplify logic operations and reduce errors in practice.[1] 
> 
> Many devices indicate position by closing and opening switches. If that device uses natural binary codes, positions 3 and 4 are next to each other but all three bits of the binary representation differ:
> 
> | Decimal  | Binary  |
> | -------- | ------- |
> | ...      | ...     |
> | 3 	     | 011     |
> | 4 	     | 100     |
> | ...      | ...     |
> 
> The problem with natural binary codes is that physical switches are not ideal: it is very unlikely that physical switches will change states exactly in synchrony. In the transition between the two states shown above, all three switches change state. In the brief period while all are changing, the switches will read some spurious position. Even without keybounce, the transition might look like 011 — 001 — 101 — 100. When the switches appear to be in position 001, the observer cannot tell if that is the "real" position 1, or a transitional state between two other positions. If the output feeds into a sequential system, possibly via combinational logic, then the sequential system may store a false value.
> 
> This problem can be solved by changing only one switch at a time, so there is never any ambiguity of position, resulting in codes assigning to each of a contiguous set of integers, or to each member of a circular list, a word of symbols such that no two code words are identical and each two adjacent code words differ by exactly one symbol. These codes are also known as unit-distance,[2][3][4][5][6] single-distance, single-step, monostrophic[7][8][5][6] or syncopic codes,[7] in reference to the Hamming distance of 1 between adjacent codes."

In [22]:
import scala.math.pow

// create a module for a GrayCoder which reorders the inputs into gray Code
class GrayCoder(bitwidth: Int) extends Module {
  val io = IO(new Bundle{
    val in = Input(UInt(bitwidth.W))
    val out = Output(UInt(bitwidth.W))
    val encode = Input(Bool())                  // decode on false
  })
  
  when (io.encode) { 
    //encode
    io.out := io.in ^ (io.in >> 1.U)            // ^ does bitwise XOR, >> is static or arithmetic right shift (in this case by 1)
  } .otherwise { 
    // decode, much more complicated
    io.out := Seq.fill(log2Ceil(bitwidth))(Wire(UInt(bitwidth.W))).zipWithIndex.fold((io.in, 0)){
      case ((w1: UInt, i1: Int), (w2: UInt, i2: Int)) => {
        w2 := w1 ^ (w1 >> pow(2, log2Ceil(bitwidth)-i2-1).toInt.U)
        (w2, i1)
      }
    }._1
  }
}


import scala.math.pow

// create a module for a GrayCoder which reorders the inputs into gray Code

defined class GrayCoder

Give it a whirl!

In [23]:
// test our gray coder
val bitwidth = 4
test(new GrayCoder(bitwidth)) { c =>
    def toBinary(i: Int, digits: Int = 8) = {
        String.format("%" + digits + "s", i.toBinaryString).replace(' ', '0')
    }
    println("Encoding:")
    for (i <- 0 until pow(2, bitwidth).toInt) {
        c.io.in.poke(i.U)
        c.io.encode.poke(true.B)
        c.clock.step(1)
        println(s"In = ${toBinary(i, bitwidth)}, Out = ${toBinary(c.io.out.peek().litValue.toInt, bitwidth)}")
    }

    println("Decoding:")
    for (i <- 0 until pow(2, bitwidth).toInt) {
        c.io.in.poke(i.U)
        c.io.encode.poke(false.B)
        c.clock.step(1)
        println(s"In = ${toBinary(i, bitwidth)}, Out = ${toBinary(c.io.out.peek().litValue.toInt, bitwidth)}")
    }
}
println("SUCCESS!!")

Elaborating design...
Done elaborating.
Encoding:
In = 0000, Out = 0000
In = 0001, Out = 0001
In = 0010, Out = 0011
In = 0011, Out = 0010
In = 0100, Out = 0110
In = 0101, Out = 0111
In = 0110, Out = 0101
In = 0111, Out = 0100
In = 1000, Out = 1100
In = 1001, Out = 1101
In = 1010, Out = 1111
In = 1011, Out = 1110
In = 1100, Out = 1010
In = 1101, Out = 1011
In = 1110, Out = 1001
In = 1111, Out = 1000
Decoding:
In = 0000, Out = 0000
In = 0001, Out = 0001
In = 0010, Out = 0011
In = 0011, Out = 0010
In = 0100, Out = 0111
In = 0101, Out = 0110
In = 0110, Out = 0100
In = 0111, Out = 0101
In = 1000, Out = 1111
In = 1001, Out = 1110
In = 1010, Out = 1100
In = 1011, Out = 1101
In = 1100, Out = 1000
In = 1101, Out = 1001
In = 1110, Out = 1011
In = 1111, Out = 1010
test GrayCoder Success: 0 tests passed in 34 cycles in 0.053280 seconds 638.14 Hz
SUCCESS!!


bitwidth: Int = 4

Gray codes are often used in asynchronous interfaces. Usually Gray counters are used rather than fully-featured encoders/decoders, but we'll use the above module to simplify things. Below is an example AsyncFIFO, built using the above Gray coder. The control logic and tester is left as an exercise for later on. For now, look at how the Gray coder is instantiated multiple times and connected.

In [24]:
class AsyncFIFO(depth: Int = 16) extends Module {
  val io = IO(new Bundle{
    // write inputs
    val write_clock = Input(Clock())    // Async!
    val write_enable = Input(Bool())
    val write_data = Input(UInt(32.W))
    
    // read inputs/outputs
    val read_clock = Input(Clock())     // Async!
    val read_enable = Input(Bool())
    val read_data = Output(UInt(32.W))
    
    // FIFO status
    val full = Output(Bool())
    val empty = Output(Bool())
  })
  
  // add extra bit to counter to check for fully/empty status (depth*2 instead of just depth as max value)
  // Counter returns a tuple whoose members can be accessed with
  // _1 for current count value and 
  // _2 for boolean indicating wrap-around
  assert(isPow2(depth), "AsyncFIFO needs a power-of-two depth!")
  val write_counter = withClock(io.write_clock) {       // use write clock
    Counter(io.write_enable && !io.full, depth*2)._1    // increment counter when writing is enabled and the Queue is not full 
    } 
  val read_counter = withClock(io.read_clock) {         // use read clock
    Counter(io.read_enable && !io.empty, depth*2)._1    // increment counter when reading is enabled and the Queue is not empty 
    }

  
  // encode
  val encoder = new GrayCoder(write_counter.getWidth)
  encoder.io.in := write_counter
  encoder.io.encode := true.B
  
  // synchronize
  val sync = withClock(io.read_clock) { ShiftRegister(encoder.io.out, 2) }
  
  // decode
  val decoder = new GrayCoder(read_counter.getWidth)
  decoder.io.in := sync
  decoder.io.encode := false.B
  
  // status logic goes here
  
}

defined class AsyncFIFO

---
# You're done!

[Return to the top.](#top)